## Load Packages

## Loading Data

In [ ]:
df = pl.read_csv(
    "data/combined_metadata_noncancer_removed.csv",
    schema_overrides={"group": pl.Utf8},   # <- dict of {col_name: dtype}
    infer_schema_length=0,                  # scan entire file for other cols
)


In [ ]:
df.head()

## Performing regex

In [10]:
# Choose text columns that exist in your DF
TEXT_COLS = [
    "title","study_name","library_name","library_strategy","library_source",
    "library_selection","tissue","dev_stage","breed","organization_name",
    "organism_scientific_name","submitter_id","biosample","bioproject",
    "experiment_alias","experiment_accession","study_accession",
    "sample_accession","platform_instrument_model","sex"
]
TEXT_COLS = [c for c in TEXT_COLS if c in df.columns]

def norm_expr(col):
    return (
        pl.col(col).cast(pl.Utf8).fill_null("")
        .str.to_lowercase()
        .str.replace_all(r"[_/|]", " ")
        .str.replace_all(r"\s+", " ")
        .str.strip_chars()          # <-- use this instead of .str.strip()
    )

exprs = [norm_expr(c) for c in TEXT_COLS]
df = df.with_columns(
    pl.concat_str(exprs, separator=" ").alias("__text") if exprs else pl.lit("").alias("__text")
)

# Regex lists
CANCER_POS = r"(?:\bcancers?\b|\btumou?rs?\b|\bmalignan(?:t|cy)\b|\bcarcinomas?\b|\bneoplasms?\b|\bmetasta(?:s|t)es?\b|\badenocarcinomas?\b|\bsarcomas?\b|\bleukemi(?:a|as)\b|\blymphom(?:a|as)\b|\bglioblastomas?\b|\bmelanomas?\b|\boncolog(?:y|ic|ical)\b)"
STRONG_POS = r"(?:\bcancers?\b|\btumou?rs?\b|\bcarcinomas?\b|\bmalignan(?:t|cy)\b|\bneoplasms?\b|\bmetasta(?:s|t)es?\b)"
CANCER_NEG = r"(?:\bnormal\b|\bhealthy\b|\bcontrols?\b|\bctrl\b|\badjacent normal\b|\bnon[-\s]?tumou?r(?:al)?\b|\bbenign\b|\bnon[-\s]?cancer(?:ous)?\b|\bsham\b|\bunaffected\b)"
NEG_SCOPED = r"(?:\b(?:no|not|without|free of)\s+(?:any\s+)?(?:cancer|tumou?r|carcinoma|malignancy|neoplasm)s?\b)"
ONCO_TRAPS = r"(?:\boncophora\b|\boncorhynchus\b|\boncotic\b|\boncomodulin\b)"

# Define your hit expressions once
pos_hit     = pl.col("__text").str.contains(CANCER_POS)
strong_pos  = pl.col("__text").str.contains(STRONG_POS)
neg_hit     = pl.col("__text").str.contains(CANCER_NEG) | pl.col("__text").str.contains(NEG_SCOPED)
onco_traps  = pl.col("__text").str.contains(ONCO_TRAPS)

# Reuse a single score expression
score_expr = (
    pos_hit.cast(pl.Int8)
    + strong_pos.cast(pl.Int8)
    - (neg_hit.cast(pl.Int8) * 2)
    - (onco_traps.cast(pl.Int8) * 2)
)

df = df.with_columns([
    score_expr.alias("cancer_score"),
    pl.when(score_expr >= 1).then(pl.lit(True))
     .when(score_expr <= -1).then(pl.lit(False))
     .otherwise(None)
     .alias("predicted_cancer"),

    # Build "explain" with concat_str instead of chained .str.concat(...)
    pl.concat_str(
        [
            pl.when(onco_traps).then(pl.lit("onco-trap")).otherwise(pl.lit("")),
            pl.when(neg_hit).then(pl.lit(",neg-context")).otherwise(pl.lit("")),
            pl.when(strong_pos).then(pl.lit(",strong-cancer-term"))
             .otherwise(pl.when(pos_hit).then(pl.lit(",weak-cancer-term")).otherwise(pl.lit(""))),
        ],
        separator=""
    )
    .str.replace_all(r"^,", "")
    .str.replace_all(r",$", "")
    .alias("explain"),
])

# Example: rows predicted non-cancer (unchanged)
non_cancer_df = df.filter(
    (pl.col("predicted_cancer") == False) | ((pl.col("predicted_cancer").is_null()) & neg_hit)
)


In [11]:
df

experiment_alias,experiment_accession,title,study_name,sample_accession,library_name,library_strategy,library_source,library_selection,library_layout,platform_instrument_model,submitter_accession,submitter_id,organization_type,organization_name,study_accession,biosample,bioproject,organism_tax_id,organism_scientific_name,breed,dev_stage,sex,tissue,biosamplemodel,run_accession,run_total_spots,run_total_bases,run_size,run_load_done,run_is_public,run_has_tax_analysis,a_count,c_count,g_count,t_count,n_count,…,processed_data_file_3,processed_data_file_4,processed_file_5,processed_file_6,confetti_clone,time_of_harvest,lgr5_status_of_injected_cells,cancer_status,subgroups,plate_number,target_gene,disease_induction,read_alignment_program,read_quantification_program,tumor_section,status_tissue,age_sacrifice,cells_loaded,gp33_status,repeat,hematopoietic_malignancy,facs_marker,tgf_beta_treatment,ena-first-public,ena-last-update,microbiome,trail_status,recipient,venus,arrayexpress-organismpart,arrayexpress-phenotype,arrayexpress-sex,arrayexpress-species,__text,cancer_score,predicted_cancer,explain
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,…,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i8,bool,str
"""Lib2""","""SRX2358837""",null,null,"""SRS1807531""","""Lib2""","""RNA-Seq""","""TRANSCRIPTOMIC""","""RANDOM""","""SINGLE""",null,"""SRA496069""","""SUB2109767""","""institute""","""AAU""","""SRP093520""",null,"""PRJNA353983""","""9915""","""Bos indicus""","""zebu""","""not collected""","""neuter""","""horn tissue""","""Model organism or animal""","""SRR5034190""","""909364.0""","""363882457.0""","""236633098.0""","""True""","""True""","""1.0""","""90507536.0""","""90635597.0""","""91335407.0""","""91268959.0""","""134958.0""",…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,""" lib2 rna-seq transcriptomic …",0,null,""""""
"""HC_30""","""SRX4947457""","""Transcriptome of Bos indicus:b…","""Illumina TruSeq mRNA Strensed …","""SRS3989751""","""HC_30""","""RNA-Seq""","""TRANSCRIPTOMIC""","""PCR""","""PAIRED""","""Illumina MiSeq""","""SRA798653""","""SUB4680127""","""institute""","""Anand Agricultural University""","""SRP167079""","""SAMN10273298""","""PRJNA497667""","""9915""","""Bos indicus""","""Kankrej""","""Myxomatous_12""","""male""","""Horn epithelial""","""Model organism or animal""","""SRR8121155""","""3671359.0""","""1138556106.0""","""719166371.0""","""True""","""True""","""1.0""","""286730289.0""","""284335813.0""","""283525767.0""","""283964235.0""","""2.0""",…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""transcriptome of bos indicus:b…",2,true,"""strong-cancer-term"""
"""HC_29""","""SRX4947456""","""Transcriptome of Bos indicus:b…","""Illumina TruSeq mRNA Strensed …","""SRS3989750""","""HC_29""","""RNA-Seq""","""TRANSCRIPTOMIC""","""PCR""","""PAIRED""","""Illumina MiSeq""","""SRA798653""","""SUB4680127""","""institute""","""Anand Agricultural University""","""SRP167079""","""SAMN10273297""","""PRJNA497667""","""9915""","""Bos indicus""","""Kankrej""","""Myxomatous_11""","""male""","""Horn epithelial""","""Model organism or animal""","""SRR8121156""","""4806102.0""","""1571499249.0""","""1023719372.0""","""True""","""True""","""1.0""","""386343262.0""","""401887678.0""","""399494765.0""","""383773540.0""","""4.0""",…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""transcriptome of bos indicus:b…",2,true,"""strong-cancer-term"""
"""HC_26""","""SRX4947455""","""Transcriptome of Bos indicus:

In [12]:
non_cancer_df

experiment_alias,experiment_accession,title,study_name,sample_accession,library_name,library_strategy,library_source,library_selection,library_layout,platform_instrument_model,submitter_accession,submitter_id,organization_type,organization_name,study_accession,biosample,bioproject,organism_tax_id,organism_scientific_name,breed,dev_stage,sex,tissue,biosamplemodel,run_accession,run_total_spots,run_total_bases,run_size,run_load_done,run_is_public,run_has_tax_analysis,a_count,c_count,g_count,t_count,n_count,…,processed_data_file_3,processed_data_file_4,processed_file_5,processed_file_6,confetti_clone,time_of_harvest,lgr5_status_of_injected_cells,cancer_status,subgroups,plate_number,target_gene,disease_induction,read_alignment_program,read_quantification_program,tumor_section,status_tissue,age_sacrifice,cells_loaded,gp33_status,repeat,hematopoietic_malignancy,facs_marker,tgf_beta_treatment,ena-first-public,ena-last-update,microbiome,trail_status,recipient,venus,arrayexpress-organismpart,arrayexpress-phenotype,arrayexpress-sex,arrayexpress-species,__text,cancer_score,predicted_cancer,explain
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,…,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i8,bool,str
"""HN_05""","""SRX4947437""","""Transcriptome of Bos indicus:b…","""Illumina TruSeq mRNA Strensed …","""SRS3989732""","""HN_05""","""RNA-Seq""","""TRANSCRIPTOMIC""","""PCR""","""PAIRED""","""Illumina MiSeq""","""SRA798653""","""SUB4680127""","""institute""","""Anand Agricultural University""","""SRP167079""","""SAMN10273273""","""PRJNA497667""","""9915""","""Bos indicus""","""Kankrej""","""Normal_5""","""male""","""Horn epithelial""","""Model organism or animal""","""SRR8121175""","""4185024.0""","""1364073713.0""","""809665392.0""","""True""","""True""","""1.0""","""343810532.0""","""339921123.0""","""340441501.0""","""339900557.0""","""0.0""",…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""transcriptome of bos indicus:b…",0,null,"""neg-context,strong-cancer-term"""
"""HN_01""","""SRX4947433""","""Transcriptome of Bos indicus:b…","""Illumina TruSeq mRNA Strensed …","""SRS3989728""","""HN_01""","""RNA-Seq""","""TRANSCRIPTOMIC""","""PCR""","""PAIRED""","""Illumina MiSeq""","""SRA798653""","""SUB4680127""","""institute""","""Anand Agricultural University""","""SRP167079""","""SAMN10273269""","""PRJNA497667""","""9915""","""Bos indicus""","""Kankrej""","""Normal_1""","""male""","""Horn epithelial""","""Model organism or animal""","""SRR8121179""","""3307344.0""","""1137880294.0""","""665972560.0""","""True""","""True""","""1.0""","""292039389.0""","""277758017.0""","""278194815.0""","""289888073.0""","""0.0""",…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""transcriptome of bos indicus:b…",0,null,"""neg-context,strong-cancer-term"""
"""HN_02""","""SRX4947432""","""Transcriptome of Bos indicus:b…","""Illumina TruSeq mRNA Strensed …","""SRS3989727""","""HN_02""","""RNA-Seq""","""TRANSCRIPTOMIC""","""PCR""","""PAIRED""","""Illumina MiSeq""","""SRA798653""","""SUB4680127""","""institute""","""Anand Agricultural University""","""SRP167079""","""SAMN10273270""","""PRJNA497667""","""9915""","""Bos indicus""","""Kankrej""","""Normal_2""","""male""","""Horn epithelial""","""Model organism or animal""","""SRR8121180""","""3832312.0""","""1320456326.0""","""781337901.0""","""True""","""True""","""1.0""","""342291385.0""","""319380214.0""","""319137687.0""","""339647040.0""","""0.0""",…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null